# E8B — the functional exact-cospectral challenge, n = 16

E2C certified that every cospectral pair receives a distinct QuIC vector.
The external review's central correction stands on our own E8A result:
C6 ties in every cospectral group, the tr(A^6) identity then forces the
diamond count to tie as well, so the E2R diamond improvement demonstrates
gain over a **linear spectral readout**, not over the spectrum itself.
The missing evidence is functional: do QuIC's within-group differences
**predict** the invariants that actually vary within exact cospectral
classes? E8A established those invariants as the Wiener index (25 varying
groups), automorphism group order (14), and radius (4, exhibit only).

**The task.** For each pair of graphs inside a varying cospectral group,
predict sign(y_i − y_j) from the difference of truncated sorted QuIC
vectors, p_k(G_i) − p_k(G_j). The adjacency spectrum is identical within
every group by exact integer census, so it is chance on this task **by
construction**, and it runs below as a mandatory control.

**Protocol, pre-registered.**
- Leave-one-group-out over the varying groups; every member and every
  pair of a group stays in one fold.
- Both orientations of every training pair: (p_i − p_j, +1) and
  (p_j − p_i, −1).
- L2 logistic regression with fit_intercept = False, which makes the
  decision function exactly antisymmetric, so a pair's correctness does
  not depend on orientation (asserted in code).
- One scalar root-mean-square rescale of the features, fit on training
  pairs only. Difference vectors have magnitude 1e-8 to 1e-5, and an
  unscaled optimizer would converge to a numerically meaningless weight
  vector; a single positive scalar preserves antisymmetry and geometry.
  No centering, no per-feature scaling, no dimensionality reduction.
- Regularization C selected on training groups only, by group-wise inner
  cross-validation with min(5, number of training groups) folds.
- Truncation arms k = 100, 400, 1000, and the full vector.
- Primary statistic: correctly ranked base pairs, with decision ties
  counted as failures (conservative). Exact one-sided binomial test
  against 0.5 and a Clopper–Pearson 95% interval. Balanced accuracy over
  canonical orientations is reported alongside.

**Permutation significance is the binomial test here, and the notebook
checks the claim empirically as a leakage detector.** Under the sign-flip
null hypothesis, every base pair's label is an independent fair coin, and
the held-out labels are independent of the predictions (which are
functions of training folds only, by the group discipline). Conditioning
on the predictions, the correct-pair count is therefore exactly
Binomial(n, 1/2), whatever the model does under training-label flips —
the model does change, and an earlier draft's claim that it stays
invariant was wrong and was caught by this notebook's own verification
cell during the dry run. The permutation-null cell below runs full
sign-flip refits and asserts the empirical null matches the binomial;
any cross-group information leak in the pipeline would inflate it.

**Pre-registered outcomes, both branches written before results.** A
positive Wiener result closes the paper's main logical gap: certified
non-spectral separation becomes functionally useful non-spectral
separation, on a task where the spectrum is chance by construction. A
negative result is reported as: QuIC contains a certified non-spectral
residue whose downstream utility remains unresolved on these censuses.
Radius is an exhibit either way (4 groups decide nothing). The exact
significance threshold for the realized pair count is computed and
printed before any accuracy appears.

**Runtime.** Minutes. The classifier fits are on tens of samples; the
automorphism enumeration over 83 graphs is seconds. If the varying-group
asserts (25 / 14 / 4) fail, the automorphism or distance computation
differs from E8A's and must be reconciled before results are read.

## Environment

In [1]:
import pickle

from collections import defaultdict
from itertools import combinations

import numpy as np
import networkx as nx

from scipy import stats

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

from tqdm.notebook import tqdm

from scipy.linalg import LinAlgWarning
import warnings
warnings.filterwarnings('ignore', category=LinAlgWarning)

In [2]:
SEED = 0
NUM_VERTICES = 16
CENSUS_BASE = "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/"

EXPECTED_GRAPH_COUNT = 4060
EXPECTED_GROUP_COUNT = 41
EXPECTED_MEMBER_COUNT = 83
EXPECTED_GROUP_SIZES = {2: 40, 3: 1}
EXPECTED_TRIPLE = [582, 3061, 3800]

# E8A gate-passer counts: groups whose members differ on each target.
EXPECTED_VARYING_GROUPS = {'wiener': 25, 'log2_aut': 14, 'radius': 4}

TRUNCATION_ARMS = (100, 400, 1000, 1 << NUM_VERTICES)
REGULARIZATION_GRID = np.logspace(-3, 3, 7)
MAX_INNER_FOLDS = 5
NUM_PERMUTATION_REFITS = 500

## Load the census and recompute the exact cospectral groups

The exact integer trace-tuple census (the E8A method) is recomputed here
because it costs seconds, and the results are hard-asserted against the
E8A numbers: 41 groups, 83 members, group sizes {2: 40, 3: 1}, the triple
present.

In [3]:
with open(CENSUS_BASE + f"n{NUM_VERTICES}_data_dict.pkl", 'rb') as census_file:
    census_data = pickle.load(census_file)
graph_keys = sorted(census_data)
assert len(graph_keys) == EXPECTED_GRAPH_COUNT

adjacency_matrices = np.stack(
    [census_data[graph_key]['adj_mat'] for graph_key in graph_keys]
).astype(np.int64)

# exact int64 trace tuples tr(A^1)..tr(A^n); walk counts stay far below int64
trace_tuples = np.zeros((EXPECTED_GRAPH_COUNT, NUM_VERTICES), dtype=np.int64)
walk_matrix = adjacency_matrices.copy()
trace_tuples[:, 0] = np.trace(walk_matrix, axis1=1, axis2=2)
for power_index in range(1, NUM_VERTICES):
    walk_matrix = walk_matrix @ adjacency_matrices
    trace_tuples[:, power_index] = np.trace(walk_matrix, axis1=1, axis2=2)

groups_by_tuple = defaultdict(list)
for graph_index in range(EXPECTED_GRAPH_COUNT):
    groups_by_tuple[tuple(trace_tuples[graph_index])].append(graph_index)
COSPECTRAL_GROUPS = sorted(sorted(member_list)
                           for member_list in groups_by_tuple.values()
                           if len(member_list) > 1)
GROUP_MEMBERS = sorted({graph_index for group in COSPECTRAL_GROUPS
                        for graph_index in group})

from collections import Counter
assert len(COSPECTRAL_GROUPS) == EXPECTED_GROUP_COUNT
assert len(GROUP_MEMBERS) == EXPECTED_MEMBER_COUNT
assert (Counter(len(group) for group in COSPECTRAL_GROUPS)
        == Counter(EXPECTED_GROUP_SIZES))
assert EXPECTED_TRIPLE in COSPECTRAL_GROUPS
print(f'{len(COSPECTRAL_GROUPS)} groups, {len(GROUP_MEMBERS)} members '
      f'— census matches E8A exactly')

# sorted QuIC vectors for the members only
MEMBER_VECTORS = {graph_index: census_data[graph_keys[graph_index]]['exact_vector']
                  for graph_index in GROUP_MEMBERS}
for graph_index, sorted_vector in MEMBER_VECTORS.items():
    assert np.all(np.diff(sorted_vector) <= 0)
    assert abs(sorted_vector.sum() - 1.0) < 1e-12

41 groups, 83 members — census matches E8A exactly


## Targets for the 83 members

The Wiener index and radius come from one all-pairs shortest-path pass
per graph (the E8A and E6 method). The automorphism group order is
computed by explicit enumeration of self-isomorphisms, which is auditable
and fast at these sizes; log base two is the regression-friendly scale
established in E2R. The varying-group counts must reproduce E8A's gate
exactly — a mismatch means the target computation differs from E8A's and
must be reconciled before any result is read.

In [4]:
def metric_targets(graph):
    """Wiener index and radius from one all-pairs shortest-path pass."""
    shortest_paths = dict(nx.all_pairs_shortest_path_length(graph))
    all_distances = [distance
                     for source_vertex in shortest_paths
                     for distance in shortest_paths[source_vertex].values()]
    eccentricities = [max(shortest_paths[source_vertex].values())
                      for source_vertex in shortest_paths]
    return sum(all_distances) // 2, min(eccentricities)


def automorphism_group_order(graph):
    """|Aut(G)| by explicit enumeration of self-isomorphisms."""
    matcher = nx.isomorphism.GraphMatcher(graph, graph)
    return sum(1 for _ in matcher.isomorphisms_iter())


MEMBER_TARGETS = {}
for graph_index in tqdm(GROUP_MEMBERS, desc='member targets'):
    member_graph = nx.from_numpy_array(adjacency_matrices[graph_index])
    wiener_index, graph_radius = metric_targets(member_graph)
    MEMBER_TARGETS[graph_index] = {
        'wiener': float(wiener_index),
        'radius': float(graph_radius),
        'log2_aut': float(np.log2(automorphism_group_order(member_graph))),
    }

TARGET_NAMES = ('wiener', 'log2_aut', 'radius')
VARYING_GROUPS = {}
for target_name in TARGET_NAMES:
    VARYING_GROUPS[target_name] = [
        group for group in COSPECTRAL_GROUPS
        if len({MEMBER_TARGETS[graph_index][target_name]
                for graph_index in group}) > 1]
    observed_count = len(VARYING_GROUPS[target_name])
    assert observed_count == EXPECTED_VARYING_GROUPS[target_name], (
        f'{target_name}: {observed_count} varying groups vs E8A '
        f'{EXPECTED_VARYING_GROUPS[target_name]} — target computation '
        f'differs from E8A; reconcile before reading results')
    print(f'{target_name}: {observed_count} varying groups (matches E8A)')

member targets:   0%|          | 0/83 [00:00<?, ?it/s]

wiener: 25 varying groups (matches E8A)
log2_aut: 14 varying groups (matches E8A)
radius: 4 varying groups (matches E8A)


## Base pairs

A base pair is an unordered pair inside a varying group whose target
values differ; equal-valued pairs inside a varying triple are excluded
and counted. Each base pair carries one canonical orientation for
bookkeeping; the classifier's antisymmetry makes correctness
orientation-independent. The exact one-sided binomial significance
threshold for each realized pair count prints here, before any accuracy
exists.

In [5]:
BASE_PAIRS = {}
for target_name in TARGET_NAMES:
    base_pair_list = []
    excluded_tie_count = 0
    for group_number, group in enumerate(VARYING_GROUPS[target_name]):
        for first_index, second_index in combinations(group, 2):
            value_difference = (MEMBER_TARGETS[first_index][target_name]
                                - MEMBER_TARGETS[second_index][target_name])
            if value_difference == 0:
                excluded_tie_count += 1
                continue
            base_pair_list.append({
                'group_number': group_number,
                'first_index': first_index,
                'second_index': second_index,
                'label': float(np.sign(value_difference)),
                'value_gap': abs(value_difference)})
    BASE_PAIRS[target_name] = base_pair_list
    pair_count = len(base_pair_list)
    smallest_significant = int(stats.binom.isf(0.05, pair_count, 0.5)) + 1
    print(f'{target_name}: {pair_count} base pairs '
          f'({excluded_tie_count} within-group ties excluded); '
          f'one-sided p < 0.05 needs >= {smallest_significant}/{pair_count} correct')

wiener: 26 base pairs (1 within-group ties excluded); one-sided p < 0.05 needs >= 18/26 correct
log2_aut: 14 base pairs (0 within-group ties excluded); one-sided p < 0.05 needs >= 11/14 correct
radius: 4 base pairs (0 within-group ties excluded); one-sided p < 0.05 needs >= 5/4 correct


## The ranking pipeline

`evaluate_target_arm` runs the full leave-one-group-out protocol for one
target and one truncation depth and returns per-pair records. The
antisymmetry of the decision function is asserted on every evaluated
pair.

In [6]:
def build_difference_features(base_pair_list, truncation_depth,
                              vector_lookup):
    """Canonical-orientation difference vectors at the given depth."""
    return np.vstack(
        [vector_lookup[base_pair['first_index']][:truncation_depth]
         - vector_lookup[base_pair['second_index']][:truncation_depth]
         for base_pair in base_pair_list])


def fit_ranking_classifier(train_features, train_labels, train_groups,
                           regularization_grid, random_state=SEED):
    """Both-orientation logistic ranker with group-wise inner selection of C.

    The training multiset receives both orientations of every pair, the
    scalar root-mean-square scale comes from the training features alone,
    and fit_intercept=False keeps the decision function antisymmetric.
    Returns (fitted_model, feature_scale, chosen_C).
    """
    feature_scale = np.sqrt((train_features ** 2).mean())
    both_orientation_features = np.vstack([train_features,
                                           -train_features]) / feature_scale
    both_orientation_labels = np.concatenate([train_labels, -train_labels])
    both_orientation_groups = np.concatenate([train_groups, train_groups])

    unique_group_count = len(set(train_groups.tolist()))
    inner_fold_count = min(MAX_INNER_FOLDS, unique_group_count)

    if inner_fold_count >= 2:
        inner_scores = []
        for candidate_C in regularization_grid:
            fold_accuracies = []
            inner_splitter = GroupKFold(n_splits=inner_fold_count)
            for inner_train, inner_test in inner_splitter.split(
                    both_orientation_features, both_orientation_labels,
                    both_orientation_groups):
                candidate_model = LogisticRegression(
                    C=candidate_C, fit_intercept=False, max_iter=5000,
                    random_state=random_state)
                candidate_model.fit(both_orientation_features[inner_train],
                                    both_orientation_labels[inner_train])
                fold_accuracies.append(
                    candidate_model.score(both_orientation_features[inner_test],
                                          both_orientation_labels[inner_test]))
            inner_scores.append(np.mean(fold_accuracies))
        chosen_C = float(regularization_grid[int(np.argmax(inner_scores))])
    else:
        chosen_C = 1.0   # single training group: no inner split possible

    fitted_model = LogisticRegression(C=chosen_C, fit_intercept=False,
                                      max_iter=5000, random_state=random_state)
    fitted_model.fit(both_orientation_features, both_orientation_labels)
    return fitted_model, feature_scale, chosen_C


def evaluate_target_arm(target_name, truncation_depth,
                        vector_lookup=None, label_override=None,
                        regularization_grid=None):
    """Leave-one-group-out ranking for one target at one truncation depth.

    Returns None when the target has fewer than two varying groups, because
    leave-one-group-out then has an empty training set; callers report the
    skip explicitly. At n = 16 the E8A asserts guarantee at least four
    varying groups per target, so this guard exists for robustness and for
    fixtures at other graph orders.
    """
    if vector_lookup is None:
        vector_lookup = MEMBER_VECTORS
    if len(VARYING_GROUPS[target_name]) < 2:
        return None
    base_pair_list = BASE_PAIRS[target_name]
    difference_features = build_difference_features(base_pair_list,
                                                    truncation_depth,
                                                    vector_lookup)
    if regularization_grid is None:
        regularization_grid = REGULARIZATION_GRID
    pair_labels = np.array([base_pair['label']
                            for base_pair in base_pair_list])
    if label_override is not None:
        pair_labels = np.asarray(label_override, dtype=float)
    pair_groups = np.array([base_pair['group_number']
                            for base_pair in base_pair_list])

    per_pair_records = []
    for held_out_group in sorted(set(pair_groups.tolist())):
        train_mask = pair_groups != held_out_group
        fitted_model, feature_scale, chosen_C = fit_ranking_classifier(
            difference_features[train_mask], pair_labels[train_mask],
            pair_groups[train_mask], regularization_grid)

        test_positions = np.nonzero(~train_mask)[0]
        for pair_position in test_positions:
            scaled_feature = (difference_features[pair_position]
                              / feature_scale).reshape(1, -1)
            decision_value = float(
                fitted_model.decision_function(scaled_feature)[0])
            mirrored_value = float(
                fitted_model.decision_function(-scaled_feature)[0])
            assert abs(decision_value + mirrored_value) < 1e-9, (
                'decision function is not antisymmetric — intercept leak')
            per_pair_records.append({
                **base_pair_list[pair_position],
                'label': pair_labels[pair_position],
                'decision_value': decision_value,
                'chosen_C': chosen_C,
                'correct': (np.sign(decision_value)
                            == pair_labels[pair_position])
                           and decision_value != 0.0})
    return per_pair_records


def summarize_records(per_pair_records):
    """Accuracy, balanced accuracy, exact binomial p, Clopper-Pearson CI."""
    pair_count = len(per_pair_records)
    correct_count = sum(record['correct'] for record in per_pair_records)
    tie_count = sum(record['decision_value'] == 0.0
                    for record in per_pair_records)
    class_recalls = []
    for class_label in (+1.0, -1.0):
        class_records = [record for record in per_pair_records
                         if record['label'] == class_label]
        if class_records:
            class_recalls.append(np.mean([record['correct']
                                          for record in class_records]))
    binomial_result = stats.binomtest(correct_count, pair_count, 0.5,
                                      alternative='greater')
    confidence_interval = binomial_result.proportion_ci(0.95, method='exact')
    return {'pair_count': pair_count, 'correct_count': correct_count,
            'tie_count': tie_count,
            'accuracy': correct_count / pair_count,
            'balanced_accuracy': float(np.mean(class_recalls)),
            'binomial_p': binomial_result.pvalue,
            'ci_low': confidence_interval.low,
            'ci_high': confidence_interval.high,
            'modal_C': float(stats.mode([record['chosen_C'] for record
                                         in per_pair_records],
                                        keepdims=False).mode)}

## Permutation-null check — a leakage detector

The header's argument needs only one empirical property: under full
sign-flip permutations (training and held-out labels flipped together,
model refit each time), the distribution of the correct-pair count must
be Binomial(n, 1/2). That holds whenever the predictions carry no
information about held-out labels beyond the training folds; any
cross-group leak in the pipeline would inflate the null. Five hundred
full refits at fixed C run on the richest target at the smallest
truncation depth, and the empirical mean is asserted within five
standard errors of one half.

In [7]:
permutation_rng = np.random.default_rng(SEED)
permutation_target = max(TARGET_NAMES,
                         key=lambda name: len(VARYING_GROUPS[name]))
permutation_depth = TRUNCATION_ARMS[0]
permutation_pair_count = len(BASE_PAIRS[permutation_target])
print(f'permutation-null check on {permutation_target} at k={permutation_depth} '
      f'({permutation_pair_count} base pairs, {NUM_PERMUTATION_REFITS} refits)')

permutation_correct_counts = []
for _ in tqdm(range(NUM_PERMUTATION_REFITS), desc='sign-flip refits'):
    flipped_labels = (np.array([base_pair['label'] for base_pair
                                in BASE_PAIRS[permutation_target]])
                      * permutation_rng.choice([-1.0, 1.0],
                                               size=permutation_pair_count))
    permuted_records = evaluate_target_arm(
        permutation_target, permutation_depth,
        label_override=flipped_labels,
        regularization_grid=np.array([1.0]))
    permutation_correct_counts.append(
        sum(record['correct'] for record in permuted_records))

permutation_correct_counts = np.array(permutation_correct_counts)
empirical_mean_fraction = permutation_correct_counts.mean() / permutation_pair_count
mean_standard_error = np.sqrt(0.25 / (NUM_PERMUTATION_REFITS
                                      * permutation_pair_count))
assert abs(empirical_mean_fraction - 0.5) < 5 * mean_standard_error, (
    f'empirical permutation null mean {empirical_mean_fraction:.4f} is more '
    f'than five standard errors from one half — cross-group leakage suspected')
binomial_quantiles = stats.binom.ppf([0.05, 0.5, 0.95],
                                     permutation_pair_count, 0.5)
empirical_quantiles = np.quantile(permutation_correct_counts, [0.05, 0.5, 0.95])
print(f'empirical null mean fraction {empirical_mean_fraction:.4f} '
      f'(binomial expects 0.5000, tolerance {5*mean_standard_error:.4f})')
print(f'quantiles 5/50/95 — empirical {empirical_quantiles}, '
      f'binomial {binomial_quantiles}: no leakage detected')

permutation-null check on wiener at k=100 (26 base pairs, 500 refits)


sign-flip refits:   0%|          | 0/500 [00:00<?, ?it/s]

empirical null mean fraction 0.5066 (binomial expects 0.5000, tolerance 0.0219)
quantiles 5/50/95 — empirical [ 9. 13. 19.], binomial [ 9. 13. 17.]: no leakage detected


## Main grid — targets × truncation depths

In [8]:
RESULTS = {}
for target_name in TARGET_NAMES:
    RESULTS[target_name] = {}
    for truncation_depth in TRUNCATION_ARMS:
        per_pair_records = evaluate_target_arm(target_name, truncation_depth)
        if per_pair_records is None:
            RESULTS[target_name][truncation_depth] = {'skipped':
                'fewer than two varying groups — leave-one-group-out impossible'}
            print(f'{target_name:>9} k={truncation_depth:>6}: SKIPPED '
                  f'(fewer than two varying groups)')
            continue
        RESULTS[target_name][truncation_depth] = {
            'summary': summarize_records(per_pair_records),
            'per_pair': per_pair_records}
        summary = RESULTS[target_name][truncation_depth]['summary']
        print(f"{target_name:>9} k={truncation_depth:>6}: "
              f"{summary['correct_count']}/{summary['pair_count']} correct "
              f"(acc {summary['accuracy']:.3f}, bal {summary['balanced_accuracy']:.3f}, "
              f"p {summary['binomial_p']:.4f}, "
              f"CI [{summary['ci_low']:.3f}, {summary['ci_high']:.3f}], "
              f"ties {summary['tie_count']}, C {summary['modal_C']:g})")

   wiener k=   100: 18/26 correct (acc 0.692, bal 0.709, p 0.0378, CI [0.513, 1.000], ties 0, C 1000)
   wiener k=   400: 18/26 correct (acc 0.692, bal 0.721, p 0.0378, CI [0.513, 1.000], ties 0, C 1000)
   wiener k=  1000: 16/26 correct (acc 0.615, bal 0.606, p 0.1635, CI [0.436, 1.000], ties 0, C 0.01)
   wiener k= 65536: 14/26 correct (acc 0.538, bal 0.539, p 0.4225, CI [0.362, 1.000], ties 0, C 0.001)
 log2_aut k=   100: 14/14 correct (acc 1.000, bal 1.000, p 0.0001, CI [0.807, 1.000], ties 0, C 10)
 log2_aut k=   400: 11/14 correct (acc 0.786, bal 0.771, p 0.0287, CI [0.534, 1.000], ties 0, C 1)
 log2_aut k=  1000: 10/14 correct (acc 0.714, bal 0.688, p 0.0898, CI [0.460, 1.000], ties 0, C 0.1)
 log2_aut k= 65536: 11/14 correct (acc 0.786, bal 0.771, p 0.0287, CI [0.534, 1.000], ties 0, C 0.001)
   radius k=   100: 3/4 correct (acc 0.750, bal 0.500, p 0.3125, CI [0.249, 1.000], ties 0, C 0.001)
   radius k=   400: 4/4 correct (acc 1.000, bal 1.000, p 0.0625, CI [0.473, 1.000], tie

## Spectrum control — chance by construction

The same pipeline runs on differences of sorted eigenvalue vectors. The
groups are exactly cospectral by integer census, so these features are
zero up to floating-point eigensolver noise; the printed maximum
within-group spectral difference documents that. Accuracy near one half
is the expected reading.

In [9]:
from numpy.linalg import eigvalsh

MEMBER_SPECTRA = {graph_index: np.sort(eigvalsh(
                      adjacency_matrices[graph_index].astype(np.float64)))
                  for graph_index in GROUP_MEMBERS}

largest_within_group_spectral_gap = max(
    np.abs(MEMBER_SPECTRA[first_index] - MEMBER_SPECTRA[second_index]).max()
    for group in COSPECTRAL_GROUPS
    for first_index, second_index in combinations(group, 2))
print(f'largest within-group spectral difference: '
      f'{largest_within_group_spectral_gap:.2e} (eigensolver noise)')

SPECTRUM_CONTROL = {}
for target_name in TARGET_NAMES:
    control_records = evaluate_target_arm(target_name, NUM_VERTICES,
                                          vector_lookup=MEMBER_SPECTRA)
    if control_records is None:
        SPECTRUM_CONTROL[target_name] = {'skipped': 'fewer than two varying groups'}
        print(f'{target_name:>9} spectrum control: SKIPPED')
        continue
    SPECTRUM_CONTROL[target_name] = summarize_records(control_records)
    control_summary = SPECTRUM_CONTROL[target_name]
    print(f"{target_name:>9} spectrum control: "
          f"{control_summary['correct_count']}/{control_summary['pair_count']} "
          f"(acc {control_summary['accuracy']:.3f}, "
          f"p {control_summary['binomial_p']:.4f}, "
          f"ties {control_summary['tie_count']})")

largest within-group spectral difference: 5.77e-15 (eigensolver noise)
   wiener spectrum control: 17/26 (acc 0.654, p 0.0843, ties 0)
 log2_aut spectrum control: 9/14 (acc 0.643, p 0.2120, ties 0)
   radius spectrum control: 3/4 (acc 0.750, p 0.3125, ties 0)


## Per-pair exhibit — Wiener at the full vector

One auditable row per base pair: group, pair, target gap, decision value,
verdict. This is the supplement table; the main-paper number is the
summary line above.

In [10]:
exhibit_depth = TRUNCATION_ARMS[-1]
print(f'{"group":>6} {"i":>5} {"j":>5} {"|dy|":>6} {"decision":>11} {"correct":>8}')
for record in sorted(RESULTS['wiener'][exhibit_depth]['per_pair'],
                     key=lambda row: row['group_number']):
    print(f"{record['group_number']:>6} {record['first_index']:>5} "
          f"{record['second_index']:>5} {record['value_gap']:>6.0f} "
          f"{record['decision_value']:>+11.4f} {str(record['correct']):>8}")

 group     i     j   |dy|    decision  correct
     0     1     5      2     +7.6693    False
     1    82   885      1     -0.0576    False
     2    83   884      1     -0.0521    False
     3   106   632      1     +1.9209    False
     4   282  1494      1     +0.0007     True
     5   457   516      1     -0.6158     True
     6   494   568      1     +0.6303     True
     7   582  3061      1     +0.2724     True
     7  3061  3800      1     -6.9901     True
     8   583  3067      1     -1.0780    False
     9   586  1502      2     +0.3545     True
    10   710   711      2     +5.8799    False
    11  1075  1086      1     -0.0554    False
    12  1371  1792      1     +0.7037     True
    13  1496  1570      1     -0.0134     True
    14  1505  1506      1     +0.4336    False
    15  1710  1734      1     +8.9436     True
    16  1898  3951      2     -7.3636     True
    17  2121  3077      1    -35.4847    False
    18  2209  2226      1     +0.4394    False
    19  2276 

In [11]:
output_payload = {
    'config': {'seed': SEED, 'truncation_arms': TRUNCATION_ARMS,
               'regularization_grid': REGULARIZATION_GRID,
               'max_inner_folds': MAX_INNER_FOLDS},
    'groups': COSPECTRAL_GROUPS,
    'member_targets': MEMBER_TARGETS,
    'base_pairs': BASE_PAIRS,
    'results': RESULTS,
    'spectrum_control': SPECTRUM_CONTROL,
    'permutation_null_check': {
        'target': permutation_target,
        'truncation_depth': int(permutation_depth),
        'refits': NUM_PERMUTATION_REFITS,
        'empirical_mean_fraction': float(empirical_mean_fraction),
        'correct_counts': permutation_correct_counts.tolist()},
}
with open('/kaggle/working/e8b_cospectral_challenge.pkl', 'wb') as output_file:
    pickle.dump(output_payload, output_file)
print('saved e8b_cospectral_challenge.pkl')

saved e8b_cospectral_challenge.pkl


## Results

(Written after the run. The reading order: (1) the census, varying-group,
and invariance asserts, which passing silently is what licenses the
binomial p-values; (2) the Wiener rows against the printed significance
threshold — the pre-registered branches are written above and the result
selects one, with the truncation arms read as where in the sorted vector
the functional signal lives, against the E2C family structure; (3) the
log2|Aut| rows, read the same way at lower power; (4) the spectrum
control at chance with numerically zero features, which is the sentence
that makes the task meaningful; (5) radius as an exhibit only. Verb
discipline: QuIC differences "rank" or "fail to rank" the invariant
within exact cospectral classes; nothing here "predicts" beyond the
censuses, and a positive result is scoped to these graph orders.)